<a href="https://colab.research.google.com/github/LennartRedlich/Capstone-Project-/blob/Anh/src/notebooks/Feature_engineering_Seniority.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Feature engineering and conventional machine learning - Domain**
In this approach, we focus on feature engineering combined with conventional machine learning models. Instead of relying primarily on textual representations, meaningful structured features are derived from the LinkedIn CV data, such as career history indicators and categorical job attributes. These features are then used to train standard classification models to predict domain.

## Preparing Data Set


In [1]:
!pip install category_encoders
!pip install catboost
import json
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
import category_encoders as ce
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.7 MB/s eta 0:00:00


In [3]:

CURRENT_YEAR = datetime.now().year

with open("/content/linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []

def extract_year(date_str):
    if not date_str:
        return None
    try:
        return int(date_str[:4])
    except:
        return None

for person_id, cv in enumerate(cvs):

    # 1. find previous jobs
    num_prev_jobs = sum(
        1 for job_item in cv if job_item.get("status") != "ACTIVE"
    )

    # 2. find the individual's start working time
    start_years = []
    for job_item in cv:
        year = extract_year(job_item.get("startDate"))
        if year:
            start_years.append(year)

    first_year = min(start_years) if start_years else None

    # 3. label ACTIVE job
    for job in cv:
        if job.get("status") == "ACTIVE":

            total_years_experience = None
            if first_year:
                total_years_experience = CURRENT_YEAR - first_year

            jobs.append({
                **job,
                "person_id": person_id,
                "num_previous_jobs": num_prev_jobs,
                "total_years_experience": total_years_experience
            })

df_active = pd.DataFrame(jobs)
df_active.head()



,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id,num_previous_jobs,total_years_experience
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management,0,1,26
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0,1,26
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional,0,1,26
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management,0,1,26
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0,1,26


##Spliting the data set


In [40]:
X = df_active[["position","seniority","num_previous_jobs","total_years_experience"]]
y = df_active["department"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Encoding categorical variables
Initially, count encoding was applied to position and seniority to provide the model with a simple numeric representation of category frequency, helping it handle high-cardinality features without creating sparse vectors. The numeric features num_previous_jobs and total_years_experience were included to capture career progression and overall experience, which are meaningful signals for predicting the domain. Together, these features give the models both categorical context and quantitative career information in a compact form.

In [41]:

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

categorical_cols = ['position','seniority']
numeric_cols = ['num_previous_jobs','total_years_experience']

categorical_transformer = ce.CountEncoder(cols=categorical_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ], remainder='passthrough'
)



##Training the model

 The three models—Logistic Regression, Random Forest, and CatBoost—were chosen to cover a range of modeling approaches suitable for categorical data.
 - Logistic Regression serves as a simple linear baseline that is easy to interpret and can reveal whether the features contain strong linear signals.
 - Random Forest captures non-linear interactions and provides robustness against noise, making it suitable for heterogeneous CV data.
 - CatBoost was included for its ability to handle categorical features natively and perform well even with small to medium-sized datasets, which is typical for LinkedIn CVs. Together, these models offer a balanced perspective on model performance before applying more advanced feature engineering.



In [42]:

#Define models
random_forest = RandomForestClassifier(n_estimators=200, random_state=42)
CatBoost = CatBoostClassifier(
            iterations=300,
            learning_rate=0.1,
            depth=6,
            loss_function="MultiClass",
            verbose=False,
            random_seed=42
        )
Logistic_Regression = LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            solver='liblinear'
        )

# Train and evaluate
for model in [random_forest, CatBoost,Logistic_Regression]:
  complete_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)])
  complete_pipeline.fit(X_train, y_train_encoded)
  y_pred = complete_pipeline.predict(X_test)
  acc = accuracy_score(y_test_encoded, y_pred)
  print(f"{type(model).__name__} accuracy: {acc:.4f}")

RandomForestClassifier accuracy: 0.4320
CatBoostClassifier accuracy: 0.5200
LogisticRegression accuracy: 0.5440


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(


- All three models—Logistic Regression, Random Forest, and CatBoost—achieved relatively low accuracy, staying around 40-50%. This indicates that the encoded features did not adequately represent the information needed to distinguish different job domains. As a result, the models showed limited ability to generalize. These findings highlighted the need for more meaningful feature transformations and motivated the subsequent model improvement steps.

## Improving the model

**Rule-based matching and One-hot encoder** provide a simple yet effective way to leverage domain knowledge by directly linking specific keywords in job titles to seniority levels. This approach is particularly useful when the dataset is small or when certain titles strongly indicate a seniority category. By applying rule-based rules in combination with feature engineering - one-hot encoder, the model can benefit from high-precision signals that are difficult to learn automatically



In [9]:
df_department = pd.read_csv("department-v2.csv")

department_dict = (
    df_department
    .groupby("label")["text"]
    .apply(list)
    .to_dict()
)
def predict_department(pos, department_dict):
    pos = pos.lower()

    for label, texts in department_dict.items():
        for t in texts:
            if t.lower() in pos:
                return label

    return "Other"

predictions_dm = []

for pos in df_active["position"]:
    pred = predict_department(pos, department_dict)
    predictions_dm.append(pred)

df_active["predicted_department"] = predictions_dm

In [34]:
#split dataset
X_rb_dm = df_active[["seniority","predicted_department","num_previous_jobs","total_years_experience"]]
y_rb_dm = df_active["department"]

X_train_rb_dm, X_test_rb_dm, y_train_rb_dm, y_test_rb_dm = train_test_split(X_rb_dm, y_rb_dm, test_size=0.2, random_state=42)

#Encode the features
preprocessor_rb_dm = ColumnTransformer(
    transformers=[
        ("dept", OneHotEncoder(handle_unknown="ignore"), ["predicted_department"]),
        ("pos", OneHotEncoder(handle_unknown="ignore"), ["seniority"])
    ],
    remainder="passthrough"
)

#training
for model in [random_forest, CatBoost,Logistic_Regression]:
  complete_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_rb_dm),
    ('model', model)])
  complete_pipeline.fit(X_train_rb_dm, y_train_rb_dm)
  y_pred_rb_dm = complete_pipeline.predict(X_test_rb_dm)
  acc = accuracy_score(y_test_rb_dm, y_pred_rb_dm)
  print(f"{type(model).__name__} accuracy: {acc:.4f}")

RandomForestClassifier accuracy: 0.6240
CatBoostClassifier accuracy: 0.6240
LogisticRegression accuracy: 0.6640


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(


- After incorporating rule-based matching and applying one-hot encoding to the categorical features, the model performance improved significantly. All three models reached an accuracy of more than 60%, indicating that the additional domain knowledge helped capture patterns more effectively. Among the three models, LogisticRegression achieved better performance with highest accurancy of 66%
- The rule-based rules provided clear and high-precision signals, especially for job titles strongly associated with specific department.
- One-hot encoding further allowed the models to distinguish categorical information without introducing frequency-related noise.